Este archivo contiene el preprocesado y división del dataset para un modelos de clasificación para reptiles, actualmente hay 10 clases (chamaleon,crocodile_alligator, frog, gecko, iguana, lizard, salamander, snake, toad, turtle_tortoise). Y la división del dataset es en proporción 70 % para train, 20 % para validation y 10 % para test. 

In [24]:
#Importaciones de librerías necesarias
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from collections import Counter
import pandas as pd
from sklearn.model_selection import train_test_split



Creamos arrays de rutas y de su respectiva clase para cada imagen

In [ ]:
#Cargar las rutas de las imágenes y sus etiquetas
rutas = []
labels = []
data_dir = r"C:\Users\rodri\OneDrive\Documentos\8vo\Dataset"

#Recorrer las carpetas y archivos para obtener las rutas y etiquetas
for carpeta in os.listdir(data_dir):
    if os.path.isdir(os.path.join(data_dir, carpeta)):
        for imagen in os.listdir(os.path.join(data_dir, carpeta)):
            rutas.append(os.path.join(data_dir, carpeta, imagen))
            labels.append(carpeta)
            
        
print("Número total de imágenes:", len(rutas))
print("Número total de etiquetas:", len(labels))
print ("Etiquetas únicas:", set(labels))
print("Número de imágenes por etiqueta:", Counter(labels))

Número total de imágenes: 6045
Número total de etiquetas: 6045
Etiquetas únicas: {'Gecko', 'Turtle_Tortoise', 'Chameleon', 'Snake', 'Iguana', 'Lizard', 'Salamander', 'Crocodile_Alligator', 'Frog', 'Toad'}
Número de imágenes por etiqueta: Counter({'Turtle_Tortoise': 1862, 'Crocodile_Alligator': 692, 'Lizard': 500, 'Snake': 500, 'Frog': 499, 'Iguana': 499, 'Toad': 497, 'Salamander': 484, 'Gecko': 302, 'Chameleon': 210})


Tenemos un problema de desbalanceo eso significa que el modelo aprenderá a favorecer las clases con más imagenes y un mal desempeño con clases pequeñas.
Entonces tenemos que ponerle pesos específicos a las clases para que penalice más los errores en las clases pequeñas

In [ ]:
#Crear un DataFrame con las rutas y etiquetas
df = pd.DataFrame({"ruta": rutas, "clase": labels})
df.head()  

,ruta,clase
0,C:\Users\rodri\OneDrive\Documentos\8vo\Dataset...,Chameleon
1,C:\Users\rodri\OneDrive\Documentos\8vo\Dataset...,Chameleon
2,C:\Users\rodri\OneDrive\Documentos\8vo\Dataset...,Chameleon
3,C:\Users\rodri\OneDrive\Documentos\8vo\Dataset...,Chameleon
4,C:\Users\rodri\OneDrive\Documentos\8vo\Dataset...,Chameleon


Se decidió dividir dataframes para que el proceso fuera más sencillo y con la ayuda de stratify gerantizamos que haya la misma proporción de cada clase

In [ ]:
#Dividir el DataFrame en conjuntos de entrenamiento, validación y prueba
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['clase'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.333, stratify=temp_df['clase'], random_state=42)
print(len(train_df))
print(len(val_df))
print(len(test_df))

4231
1209
605


Una vez dividido todo ahora si procedemos a hacer data augmentation solo para test

In [ ]:
#Crear generadores de datos para entrenamiento, validación y prueba
train_datagen = ImageDataGenerator(rescale=1./255,
                                   zoom_range=0.3,
                                   rotation_range=20,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   horizontal_flip=True)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(train_df, x_col='ruta', y_col='clase', target_size=(224, 224), batch_size=32, class_mode='categorical')
val_generator = val_datagen.flow_from_dataframe(val_df, x_col='ruta', y_col='clase', target_size=(224, 224), batch_size=32, class_mode='categorical')
test_generator = val_datagen.flow_from_dataframe(test_df, x_col='ruta', y_col='clase', target_size=(224, 224), batch_size=32, class_mode='categorical')

print(train_generator.class_indices)

Found 4230 validated image filenames belonging to 10 classes.
Found 1209 validated image filenames belonging to 10 classes.
Found 605 validated image filenames belonging to 10 classes.
{'Chameleon': 0, 'Crocodile_Alligator': 1, 'Frog': 2, 'Gecko': 3, 'Iguana': 4, 'Lizard': 5, 'Salamander': 6, 'Snake': 7, 'Toad': 8, 'Turtle_Tortoise': 9}


c:\Users\rodri\anaconda3\envs\OpenGL\Lib\site-packages\keras\src\legacy\preprocessing\image.py:918: UserWarning: Found 1 invalid image filename(s) in x_col="ruta". These filename(s) will be ignored.
  warnings.warn(
